In [2]:
from tqdm.auto import tqdm
import pickle
from functools import partial
import numpy as np
import torch
import os
import time
from collections import defaultdict
import torch

In [2]:

meta = torch.load(os.path.join('/data/zkj-data/dataset/facerec_val/IJBC_gt_aligned', 'metadata.pt'), weights_only=False)

In [ ]:
from datasets import Dataset

dataset = Dataset.load_from_disk('/data/zkj-data/dataset/facerec_val/IJBC_gt_aligned')
dataset.column_names

/root/anaconda3/envs/cvlface/lib/python3.10/site-packages/datasets/table.py:1421: FutureWarning: promote has been superseded by promote_options='default'.
  table = cls._concat_blocks(blocks, axis=0)


['image', 'index']

In [ ]:
# index_template_dict = {i: meta['templates'][i] for i in range(len(meta['templates']))}

In [4]:
# group_map 记录了 template 与 identity 的对应关系
# index_template_dict 记录了序号 与 template 的对应关系
# negative_pairs 记录了 identity 与 identity 之间的负样本对
# identity_to_indices 记录了 identity 与对应的图片序号列表

# 负样本评估: 逐个遍历 negative_pairs, 找到对应的图片序号列表, 计算两组图片特征之间的相似度
# 正样本评估: 逐个遍历 positive_pairs, 找到对应的图片序号列表, 计算组内图片特征之间的相似度
def get_pairs_data(meta):
    def get_group_map(meta):
        # 并查集获取所有template的唯一id，包括孤岛
        # 收集所有唯一节点
        all_nodes = set(meta['p1']) | set(meta['p2'])

        class UnionFind:
            def __init__(self, nodes):
                self.parent = {x: x for x in nodes}
                self.rank = {x: 0 for x in nodes}
            
            def find(self, x):
                if self.parent[x] != x:
                    self.parent[x] = self.find(self.parent[x])
                return self.parent[x]
            
            def union(self, x, y):
                rx, ry = self.find(x), self.find(y)
                if rx == ry:
                    return
                if self.rank[rx] < self.rank[ry]:
                    rx, ry = ry, rx
                self.parent[ry] = rx
                if self.rank[rx] == self.rank[ry]:
                    self.rank[rx] += 1

        # 初始化并查集
        uf = UnionFind(all_nodes)

        # 合并 label == 1 的边
        for label, p1, p2 in zip(meta['label'], meta['p1'], meta['p2']):
            if label == 1:
                uf.union(p1, p2)

        # 生成 group_map（包含孤岛）
        group_map = {}
        root_to_id = {}
        for node in all_nodes:
            root = uf.find(node)
            if root not in root_to_id:
                root_to_id[root] = len(root_to_id)  # 自动递增 group_id
            group_map[node] = root_to_id[root]
        
        return group_map
    
    def get_negative_pairs(meta, group_map):
        # --- Step 1: 沿用已有的并查集逻辑（略）---
        # 假设此时 group_map 已构建完成：{node: group_id}

        # --- Step 2: 构建负样本对比对（基于 label == 0）---
        negative_pairs = set()  # 使用 set 自动去重

        for label, p1, p2 in zip(meta['label'], meta['p1'], meta['p2']):
            if label == 0:
                # 确保 p1 和 p2 都在 all_nodes 中（通常成立）
                if p1 in group_map and p2 in group_map:
                    gid1 = group_map[p1]
                    gid2 = group_map[p2]
                    # 理论上 gid1 != gid2，但可加断言或跳过异常
                    if gid1 != gid2:
                        # 规范化顺序：小ID在前，避免 (a,b) 和 (b,a) 重复
                        pair = (min(gid1, gid2), max(gid1, gid2))
                        negative_pairs.add(pair)
        return list(negative_pairs)
    
    def get_positive_pairs(meta, group_map):
        # 只需每个 group_id 自己与自己配对
        positive_pairs = set()  # 使用 set 自动去重

        for gid in group_map.values():
            positive_pairs.add((gid, gid))
        return list(positive_pairs)
    
    def get_identity_to_indices(meta, group_map):
        identity_to_indices = defaultdict(list)
        for idx, template in enumerate(meta['templates']):
            if template in group_map:  # 确保 template 在 group_map 中
                identity = group_map[template]
                identity_to_indices[identity].append(idx)
            else:
                print(f"Template {template} not found in group_map.")

        return identity_to_indices
    
    group_map = get_group_map(meta)
    negative_pairs = get_negative_pairs(meta, group_map)
    positive_pairs = get_positive_pairs(meta, group_map)
    identity_to_indices = get_identity_to_indices(meta, group_map)
    return group_map, negative_pairs, positive_pairs, identity_to_indices
group_map, negative_pairs, positive_pairs, identity_to_indices = get_pairs_data(meta)

In [5]:
# 现在得到了template的唯一id映射, 得到了图片序号与template的映射, 就可以得到图片与doc的映射
index_docid_dict = {i: group_map[meta['templates'][i]] for i in range(len(meta['templates']))}
index_docid_list = [index_docid_dict[i] for i in range(len(index_docid_dict))]  # 按图片序号排列的docid列表

In [6]:
def compute_tpir_from_heap(neg_heap, pos_scores, total_neg_pairs, target_fars):
    """
    从堆中计算 TPIR
    """
    # 将堆转换为排序数组（从高到低）
    neg_scores_sorted = sorted(neg_heap, reverse=True)
    
    results = {}
    thresholds = {}
    
    for far in target_fars:
        # 计算对应的索引
        idx = int(far * total_neg_pairs)
        
        if idx < len(neg_scores_sorted):
            threshold = neg_scores_sorted[idx]
        else:
            # 如果 FAR 太小，使用最小的负样本分数
            threshold = neg_scores_sorted[-1] if neg_scores_sorted else 0.0
        
        # 计算 TPIR
        tpir = np.mean(pos_scores >= threshold)if len(pos_scores) > 0 else 0.0
        
        results[f'tpir_at_far_{far}'] = float(tpir) * 100 
        thresholds[far] = threshold
    
    return results, thresholds
def compute_tpir_optimized(query_feats_list, query_ids, target_fars=[1e-5, 1e-6, 5e-7, 1e-7, 1e-8, 1e-9, 1e-10]):
    """
    优化版本：只维护 top 1e-5 的负样本分数
    """
    
    N = len(query_ids)
    device = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    feats_t = torch.tensor(query_feats_list, dtype=torch.float32).to(device)
    ids_t = torch.tensor(query_ids).to(device)

    # 预计算总的负样本数量（基于上三角矩阵）
    total_pairs = N * (N - 1) // 2  # 上三角对数
    unique_ids, counts = np.unique(query_ids, return_counts=True)
    total_pos_pairs = sum(c * (c - 1) // 2 for c in counts)  # 正样本对数
    total_neg_pairs = total_pairs - total_pos_pairs  # 负样本对数
    
    print(f"总对数: {total_pairs}, 正样本对数: {total_pos_pairs}, 负样本对数: {total_neg_pairs}")
    
    # 计算需要维护的 top-k 大小
    max_far = max(target_fars)
    top_k = max(int(total_neg_pairs * max_far), 1000)  # 至少保留1000个
    print(f"维护 top-{top_k} 负样本分数")
    
    # 使用最小堆维护 top-k 负样本分数
    neg_heap = []
    pos_scores = []
    
    block_size = 2048*5
    start = time.time()
    
    for i in tqdm(range(0, N, block_size), desc='Processing blocks'):
        block1 = feats_t[i:i+block_size]
        ids1 = ids_t[i:i+block_size]

        for j in range(i, N, block_size):
            block2 = feats_t[j:j+block_size]
            ids2 = ids_t[j:j+block_size]

            # 相似度计算
            sim_block = block1 @ block2.T

            # 标签匹配
            label_eq = (ids1[:, None] == ids2[None, :])

            # 上三角 mask
            if i == j:
                mask = torch.triu(torch.ones_like(sim_block, dtype=torch.bool), diagonal=1)
            else:
                mask = torch.ones_like(sim_block, dtype=torch.bool)

            # 提取有效的相似度和标签
            flat_sim = sim_block[mask]
            flat_labels = label_eq[mask]

            # 正样本分数直接收集
            pos_sim = flat_sim[flat_labels]
            if len(pos_sim) > 0:
                pos_scores.append(pos_sim.half().cpu())

            # 负样本分数只保留 top-k
            neg_sim = flat_sim[~flat_labels]
            k_local = min(top_k * 2, len(neg_sim))  # 取稍多一点，避免遗漏
            topk_neg = neg_sim.topk(k_local).values
            neg_candidates = topk_neg.half().cpu().numpy()  # 传到 CPU
            
            if len(neg_heap) == 0:
                neg_heap = neg_candidates
            else:
                neg_heap = np.concatenate([neg_heap, neg_candidates])
            
            if len(neg_heap) > top_k:
                # O(n) 分区操作
                neg_heap = np.partition(neg_heap, -top_k)[-top_k:]
                # 可选：排序便于后续判断最小值
                neg_heap = np.sort(neg_heap)  # 升序，最小值在 [0]
            else:
                neg_heap = np.sort(neg_heap)  # 维持有序

    print(f"计算矩阵耗时: {time.time() - start:.2f} 秒")
    
    # 转换正样本分数
    start = time.time()
    pos_scores = torch.cat(pos_scores).numpy() if pos_scores else np.array([])
    print(f"正样本处理耗时: {time.time() - start:.2f} 秒")
    print(f"正样本对数量: {len(pos_scores)}, 维护的负样本对数量: {len(neg_heap)}")

    # 计算 TPIR
    start = time.time()
    result, thresholds = compute_tpir_from_heap(neg_heap, pos_scores, total_neg_pairs, target_fars)
    print(f"计算 TPIR 耗时: {time.time() - start:.2f} 秒")
    
    return result, thresholds

In [ ]:
# epoch48 进行1:1全量对比
with open('/data/zkj-data/dataset/facerec_val/IJBC_gt_aligned/epoch48_features.pkl', 'rb') as f:
    data = pickle.load(f)
embeddings = (data['collection']['features']).numpy()
img_input_feats = embeddings.copy()
img_input_feats = img_input_feats / np.sqrt(np.sum(img_input_feats ** 2, -1, keepdims=True))
result, thresholds = compute_tpir_optimized(img_input_feats, index_docid_list)
target_fars=[1e-5, 1e-6, 5e-7, 1e-7, 1e-8, 1e-9, 1e-10]
for far in target_fars:
    print(f"threshold_{far}: {thresholds[far]:.6f}, tpir_{far}: {result[f'tpir_at_far_{far}']:.4f}%")

总对数: 110156210625, 正样本对数: 108251342, 负样本对数: 110047959283
维护 top-1100479 负样本分数


Processing blocks:   0%|          | 0/46 [00:00<?, ?it/s]

计算矩阵耗时: 95.98 秒
正样本处理耗时: 0.06 秒
正样本对数量: 108251342, 维护的负样本对数量: 1100479
计算 TPIR 耗时: 7.47 秒
threshold_1e-05: 0.906738, tpir_1e-05: 2.9720%
threshold_1e-06: 0.958496, tpir_1e-06: 0.7648%
threshold_5e-07: 0.965820, tpir_5e-07: 0.6806%
threshold_1e-07: 0.977539, tpir_1e-07: 0.6117%
threshold_1e-08: 0.997070, tpir_1e-08: 0.5671%
threshold_1e-09: 1.000000, tpir_1e-09: 0.5661%
threshold_1e-10: 1.000000, tpir_1e-10: 0.5661%


In [21]:
# webface 进行1:1全量对比
with open('/data/zkj-data/dataset/facerec_val/IJBC_gt_aligned/webface_features.pkl', 'rb') as f:
    data = pickle.load(f)
embeddings = (data['collection']['features']).numpy()
img_input_feats = embeddings.copy()
img_input_feats = img_input_feats / np.sqrt(np.sum(img_input_feats ** 2, -1, keepdims=True))
result, thresholds = compute_tpir_optimized(img_input_feats, index_docid_list)
target_fars=[1e-5, 1e-6, 5e-7, 1e-7, 1e-8, 1e-9, 1e-10]
for far in target_fars:
    print(f"threshold_{far}: {thresholds[far]:.6f}, tpir_{far}: {result[f'tpir_at_far_{far}']:.4f}%")

总对数: 110156210625, 正样本对数: 108251342, 负样本对数: 110047959283
维护 top-1100479 负样本分数


Processing blocks:   0%|          | 0/46 [00:00<?, ?it/s]

计算矩阵耗时: 99.83 秒
正样本处理耗时: 0.07 秒
正样本对数量: 108251342, 维护的负样本对数量: 1100479
计算 TPIR 耗时: 7.52 秒
threshold_1e-05: 0.492676, tpir_1e-05: 68.1592%
threshold_1e-06: 0.694824, tpir_1e-06: 28.2479%
threshold_5e-07: 0.739746, tpir_5e-07: 19.7494%
threshold_1e-07: 0.828125, tpir_1e-07: 8.0017%
threshold_1e-08: 0.982910, tpir_1e-08: 0.5873%
threshold_1e-09: 1.000000, tpir_1e-09: 0.5661%
threshold_1e-10: 1.000000, tpir_1e-10: 0.5661%


In [7]:
# webface 进行1:1全量对比
with open('/data/zkj-data/dataset/facerec_val/IJBC_gt_aligned/s3_full_29_features.pkl', 'rb') as f:
    data = pickle.load(f)
embeddings = (data['collection']['features']).numpy()
img_input_feats = embeddings.copy()
img_input_feats = img_input_feats / np.sqrt(np.sum(img_input_feats ** 2, -1, keepdims=True))
result, thresholds = compute_tpir_optimized(img_input_feats, index_docid_list)
target_fars=[1e-5, 1e-6, 5e-7, 1e-7, 1e-8, 1e-9, 1e-10]
for far in target_fars:
    print(f"threshold_{far}: {thresholds[far]:.6f}, tpir_{far}: {result[f'tpir_at_far_{far}']:.4f}%")

总对数: 110156210625, 正样本对数: 108251342, 负样本对数: 110047959283
维护 top-1100479 负样本分数


Processing blocks:   0%|          | 0/46 [00:00<?, ?it/s]

计算矩阵耗时: 96.86 秒
正样本处理耗时: 0.07 秒
正样本对数量: 108251342, 维护的负样本对数量: 1100479
计算 TPIR 耗时: 7.62 秒
threshold_1e-05: 0.849609, tpir_1e-05: 9.9529%
threshold_1e-06: 0.917969, tpir_1e-06: 2.7039%
threshold_5e-07: 0.930176, tpir_5e-07: 1.9213%
threshold_1e-07: 0.950684, tpir_1e-07: 1.0430%
threshold_1e-08: 0.991211, tpir_1e-08: 0.5846%
threshold_1e-09: 1.000000, tpir_1e-09: 0.5661%
threshold_1e-10: 1.000000, tpir_1e-10: 0.5661%


In [8]:
# 001的图片列表
with open('./image_list2.txt', 'r') as f:
    lines = f.readlines()
image_list_001 = [line.strip() for line in lines]
image_list_001 = [int(image.split('.')[0])-1 for image in image_list_001 ]
image_list_001_set= set(image_list_001)
image_list_001 = np.array(image_list_001)


In [14]:
with open('/data/zkj-data/dataset/facerec_val/IJBC_gt_aligned/webface_features.pkl', 'rb') as f:
    data = pickle.load(f)
embeddings = (data['collection']['features']).numpy()
img_input_feats = embeddings.copy()
img_input_feats = img_input_feats / np.sqrt(np.sum(img_input_feats ** 2, -1, keepdims=True))
image_feat_001 = img_input_feats[image_list_001]
index_docid_list_001 = np.array([index_docid_list[i] for i in image_list_001])
result, thresholds = compute_tpir_optimized(image_feat_001, index_docid_list_001)
target_fars=[1e-5, 1e-6, 5e-7, 1e-7, 1e-8, 1e-9, 1e-10]
for far in target_fars:
    print(f"threshold_{far}: {thresholds[far]:.6f}, tpir_{far}: {result[f'tpir_at_far_{far}']:.4f}%")

总对数: 80264017470, 正样本对数: 88089799, 负样本对数: 80175927671
维护 top-801759 负样本分数


Processing blocks:   0%|          | 0/40 [00:00<?, ?it/s]

计算矩阵耗时: 55.81 秒
正样本处理耗时: 0.04 秒
正样本对数量: 88089799, 维护的负样本对数量: 801759
计算 TPIR 耗时: 6.21 秒
threshold_1e-05: 0.309570, tpir_1e-05: 96.2331%
threshold_1e-06: 0.548340, tpir_1e-06: 65.7789%
threshold_5e-07: 0.659180, tpir_5e-07: 41.4895%
threshold_1e-07: 0.836914, tpir_1e-07: 8.4596%
threshold_1e-08: 1.000000, tpir_1e-08: 0.5940%
threshold_1e-09: 1.000000, tpir_1e-09: 0.5940%
threshold_1e-10: 1.000000, tpir_1e-10: 0.5940%


In [15]:
with open('/data/zkj-data/dataset/facerec_val/IJBC_gt_aligned/epoch48_features.pkl', 'rb') as f:
    data = pickle.load(f)
embeddings = (data['collection']['features']).numpy()
img_input_feats = embeddings.copy()
img_input_feats = img_input_feats / np.sqrt(np.sum(img_input_feats ** 2, -1, keepdims=True))
image_feat_001 = img_input_feats[image_list_001]
index_docid_list_001 = np.array([index_docid_list[i] for i in image_list_001])
result, thresholds = compute_tpir_optimized(image_feat_001, index_docid_list_001)
target_fars=[1e-5, 1e-6, 5e-7, 1e-7, 1e-8, 1e-9, 1e-10]
for far in target_fars:
    print(f"threshold_{far}: {thresholds[far]:.6f}, tpir_{far}: {result[f'tpir_at_far_{far}']:.4f}%")

总对数: 80264017470, 正样本对数: 88089799, 负样本对数: 80175927671
维护 top-801759 负样本分数


Processing blocks:   0%|          | 0/40 [00:00<?, ?it/s]

计算矩阵耗时: 57.99 秒
正样本处理耗时: 0.04 秒
正样本对数量: 88089799, 维护的负样本对数量: 801759
计算 TPIR 耗时: 6.16 秒
threshold_1e-05: 0.496094, tpir_1e-05: 78.2805%
threshold_1e-06: 0.576660, tpir_1e-06: 64.1974%
threshold_5e-07: 0.648926, tpir_5e-07: 48.6577%
threshold_1e-07: 0.840820, tpir_1e-07: 11.2485%
threshold_1e-08: 1.000000, tpir_1e-08: 0.5940%
threshold_1e-09: 1.000000, tpir_1e-09: 0.5940%
threshold_1e-10: 1.000000, tpir_1e-10: 0.5940%


In [16]:
with open('/data/zkj-data/dataset/facerec_val/IJBC_gt_aligned/s3_full_29_features.pkl', 'rb') as f:
    data = pickle.load(f)
embeddings = (data['collection']['features']).numpy()
img_input_feats = embeddings.copy()
img_input_feats = img_input_feats / np.sqrt(np.sum(img_input_feats ** 2, -1, keepdims=True))
image_feat_001 = img_input_feats[image_list_001]
index_docid_list_001 = np.array([index_docid_list[i] for i in image_list_001])
result, thresholds = compute_tpir_optimized(image_feat_001, index_docid_list_001)
target_fars=[1e-5, 1e-6, 5e-7, 1e-7, 1e-8, 1e-9, 1e-10]
for far in target_fars:
    print(f"threshold_{far}: {thresholds[far]:.6f}, tpir_{far}: {result[f'tpir_at_far_{far}']:.4f}%")

总对数: 80264017470, 正样本对数: 88089799, 负样本对数: 80175927671
维护 top-801759 负样本分数


Processing blocks:   0%|          | 0/40 [00:00<?, ?it/s]

计算矩阵耗时: 57.64 秒
正样本处理耗时: 0.05 秒
正样本对数量: 88089799, 维护的负样本对数量: 801759
计算 TPIR 耗时: 6.27 秒
threshold_1e-05: 0.512207, tpir_1e-05: 81.4082%
threshold_1e-06: 0.602051, tpir_1e-06: 65.6787%
threshold_5e-07: 0.685547, tpir_5e-07: 46.6481%
threshold_1e-07: 0.847656, tpir_1e-07: 12.1039%
threshold_1e-08: 1.000000, tpir_1e-08: 0.5940%
threshold_1e-09: 1.000000, tpir_1e-09: 0.5940%
threshold_1e-10: 1.000000, tpir_1e-10: 0.5940%
